# Fine-Tune Falcon-7B with LoRA

Companion notebook for the To Data & Beyond tutorial. It preserves the article's complete workflow: authenticate with Hugging Face, load OpenAssistant Guanaco, quantize Falcon-7B to 4-bit, configure LoRA, define the training arguments, and fine-tune with TRL.

> **Runtime:** Use a CUDA GPU runtime. Falcon-7B does not fit comfortably in a CPU-only notebook. The training run reports metrics to Weights & Biases, so sign in when prompted or change `report_to` to `"none"`. Package APIs have evolved since the archived tutorial; the versions below preserve its original `SFTTrainer` interface.

In [ ]:
%pip install -q "transformers==4.40.2" "trl==0.8.6" "peft==0.10.0" "accelerate==0.29.3" "datasets==2.19.1" "bitsandbytes==0.43.1" "wandb==0.17.0" einops

## 1. Authenticate

Create a Hugging Face read token in your account settings and enter it only in the interactive prompt. Never paste tokens into the notebook.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import wandb

wandb.login()  # Enter the key in the prompt; do not save it in this file.

## 2. Load the dataset

In [ ]:
from datasets import load_dataset

dataset_name = "timdettmers/openassistant-guanaco"
dataset = load_dataset(dataset_name, split="train")

In [ ]:
dataset[0]

## 3. Load the quantized model and tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required for this tutorial.")

model_name = "ybelkada/falcon-7b-sharded-bf16"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

## 4. Define the LoRA configuration

In [ ]:
from peft import LoraConfig

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "query_key_value",
        "dense",
        "dense_h_to_4h",
        "dense_4h_to_h",
    ],
)

## 5. Define the training arguments

In [ ]:
from transformers import TrainingArguments

output_dir = "./results"
per_device_train_batch_size = 1
gradient_accumulation_steps = 1
optim = "paged_adamw_32bit"
save_steps = 10
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 200
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=True,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    report_to="wandb",
)

## 6. Fine-tune the model

In [ ]:
from trl import SFTTrainer

max_seq_length = 200
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
)

In [ ]:
trainer.train()

## 7. Save the LoRA adapter

Saving the adapter keeps the output much smaller than a full Falcon-7B checkpoint.

In [ ]:
adapter_dir = "./falcon-7b-openassistant-lora"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)